In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _download_range_data(ticker, weeks, buffer_weeks=8):
    end = pd.Timestamp.today().normalize()
    start = end - pd.Timedelta(weeks=weeks + buffer_weeks)
    df = yf.download(ticker, start=start, end=end, interval='1d', auto_adjust=False, progress=False)
    if df.empty:
        raise ValueError(f'No data returned for {ticker!r}.')
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    return df.copy()


def _add_range_indicators(df):
    df['SMA20'] = df['Close'].rolling(20).mean()
    df['STD20'] = df['Close'].rolling(20).std()
    df['Upper'] = df['SMA20'] + 2 * df['STD20']
    df['Lower'] = df['SMA20'] - 2 * df['STD20']
    df['BandWidth'] = (df['Upper'] - df['Lower']) / df['SMA20']
    df['PercentB'] = (df['Close'] - df['Lower']) / (df['Upper'] - df['Lower'])

    df['TR'] = np.maximum(
        df['High'] - df['Low'],
        np.maximum(
            (df['High'] - df['Close'].shift()).abs(),
            (df['Low'] - df['Close'].shift()).abs(),
        ),
    )
    df['ATR14'] = df['TR'].rolling(14).mean()
    df['RangeHigh20'] = df['High'].rolling(20).max()
    df['RangeLow20'] = df['Low'].rolling(20).min()
    df['RangeSize20'] = df['RangeHigh20'] - df['RangeLow20']
    df['ATR_to_Range'] = df['ATR14'] / df['RangeSize20']

    tr_sum = df['TR'].rolling(14).sum()
    range_14 = df['High'].rolling(14).max() - df['Low'].rolling(14).min()
    df['CHOP14'] = 100 * np.log10(tr_sum / range_14) / np.log10(14)

    df['MidRangePos'] = (df['Close'] - df['RangeLow20']) / df['RangeSize20']
    df['MidRangeDistance'] = (df['MidRangePos'] - 0.5).abs() * 2
    return df


def _normalize(series):
    valid = series.dropna()
    if valid.empty:
        return series * np.nan
    min_val = valid.min()
    max_val = valid.max()
    if max_val == min_val:
        return series * 0 + 50
    return ((series - min_val) / (max_val - min_val)) * 100


def analyze_range_boundness(ticker, weeks=12):
    if not isinstance(weeks, int) or weeks <= 0:
        raise ValueError('weeks must be a positive integer.')

    df = _download_range_data(ticker, weeks)
    df = _add_range_indicators(df)

    recent = df.tail(max(weeks * 5, 20)).copy()
    recent['LowBandWidthScore'] = 100 - _normalize(recent['BandWidth'])
    recent['LowAtrRangeScore'] = 100 - _normalize(recent['ATR_to_Range'])
    recent['HighChopScore'] = _normalize(recent['CHOP14'])
    recent['CenterScore'] = 100 - _normalize(recent['MidRangeDistance'])
    recent['RangeBoundScore'] = (
        0.35 * recent['LowBandWidthScore']
        + 0.25 * recent['LowAtrRangeScore']
        + 0.25 * recent['HighChopScore']
        + 0.15 * recent['CenterScore']
    )

    latest = recent.dropna(subset=['RangeBoundScore']).iloc[-1]
    state = 'Range-bound' if latest['RangeBoundScore'] >= 60 else 'Trending'

    stats = pd.DataFrame(
        {
            'Metric': [
                'Ticker',
                'Weeks analyzed',
                'Latest close',
                '20D range high',
                '20D range low',
                'BandWidth',
                '%B',
                'ATR / range',
                'CHOP14',
                'Range-bound score',
                'Interpretation',
            ],
            'Value': [
                ticker.upper(),
                weeks,
                latest['Close'],
                latest['RangeHigh20'],
                latest['RangeLow20'],
                latest['BandWidth'],
                latest['PercentB'],
                latest['ATR_to_Range'],
                latest['CHOP14'],
                latest['RangeBoundScore'],
                state,
            ],
        }
    )

    pivot_highs = recent[(recent['High'] > recent['High'].shift(1)) & (recent['High'] > recent['High'].shift(-1))]
    pivot_lows = recent[(recent['Low'] < recent['Low'].shift(1)) & (recent['Low'] < recent['Low'].shift(-1))]

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.5, 0.25, 0.25],
        subplot_titles=(
            f'{ticker.upper()} price and range envelope',
            'Compression indicators',
            'Range-bound score',
        ),
    )

    fig.add_trace(go.Scatter(x=recent.index, y=recent['Close'], name='Close', line=dict(color='#1f77b4', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=recent.index, y=recent['SMA20'], name='SMA 20', line=dict(color='#ff7f0e', width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=recent.index, y=recent['Upper'], name='Upper band', line=dict(color='rgba(214, 39, 40, 0.7)', width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=recent.index, y=recent['Lower'], name='Lower band', line=dict(color='rgba(44, 160, 44, 0.7)', width=1), fill='tonexty', fillcolor='rgba(44, 160, 44, 0.08)'), row=1, col=1)
    fig.add_trace(go.Scatter(x=pivot_highs.index, y=pivot_highs['High'], name='Swing high', mode='markers', marker=dict(color='crimson', size=7, symbol='triangle-up')), row=1, col=1)
    fig.add_trace(go.Scatter(x=pivot_lows.index, y=pivot_lows['Low'], name='Swing low', mode='markers', marker=dict(color='green', size=7, symbol='triangle-down')), row=1, col=1)

    fig.add_trace(go.Scatter(x=recent.index, y=recent['BandWidth'], name='BandWidth', line=dict(color='#9467bd', width=1.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=recent.index, y=recent['ATR_to_Range'], name='ATR / range', line=dict(color='#8c564b', width=1.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=recent.index, y=recent['CHOP14'], name='CHOP14', line=dict(color='#17becf', width=1.5)), row=2, col=1)

    fig.add_trace(go.Scatter(x=recent.index, y=recent['RangeBoundScore'], name='Range-bound score', line=dict(color='#2ca02c', width=2)), row=3, col=1)
    fig.add_hline(y=60, line_dash='dash', line_color='gray', row=3, col=1)

    fig.update_layout(
        height=1000,
        title=f'{ticker.upper()} range-bound analysis over the last {weeks} weeks',
        template='plotly_white',
        legend_orientation='h',
        legend_y=1.02,
        margin=dict(l=40, r=20, t=80, b=40),
    )
    fig.update_yaxes(title_text='Price', row=1, col=1)
    fig.update_yaxes(title_text='Indicator value', row=2, col=1)
    fig.update_yaxes(title_text='Score', row=3, col=1, range=[0, 100])

    display(stats.style.format({
        'Latest close': '{:,.2f}',
        '20D range high': '{:,.2f}',
        '20D range low': '{:,.2f}',
        'BandWidth': '{:.2%}',
        '%B': '{:.2f}',
        'ATR / range': '{:.2%}',
        'CHOP14': '{:.2f}',
        'Range-bound score': '{:.2f}',
    }))
    display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))

    return df, recent, stats, fig


# Adjust these two inputs and run the cell to inspect the result.
ticker = 'QQQ'
weeks = 90
analyze_range_boundness(ticker, weeks)

,Metric,Value
0,Ticker,QQQ
1,Weeks analyzed,90
2,Latest close,712.599976
3,20D range high,745.450012
4,20D range low,686.369995
5,BandWidth,0.076425
6,%B,0.345690
7,ATR / range,0.290816
8,CHOP14,65.557261
9,Range-bound score,71.549324


(             Adj Close       Close        High         Low        Open  \
 Date                                                                     
 2024-08-19  476.275116  481.269989  481.309998  473.369995  475.170013   
 2024-08-20  475.275604  480.260010  482.940002  478.549988  480.350006   
 2024-08-21  477.492371  482.500000  484.369995  479.320007  481.049988   
 2024-08-22  469.921753  474.850006  485.540009  473.809998  484.839996   
 2024-08-23  475.018341  480.000000  482.739990  475.279999  479.239990   
 ...                ...         ...         ...         ...         ...   
 2026-06-26  706.520020  706.520020  715.559998  702.809998  707.130005   
 2026-06-29  724.080017  724.080017  724.580017  705.169983  713.989990   
 2026-06-30  736.400024  736.400024  737.619995  723.909973  724.179993   
 2026-07-01  725.169983  725.169983  731.919983  724.599976  729.190002   
 2026-07-02  712.599976  712.599976  730.830017  707.559998  725.580017   
 
               Volume  